# Insurance DW — Sample Analytic Queries

This notebook demonstrates analytic queries against the DuckDB data warehouse  
`outputs/insurance_dw.duckdb`, produced by running:

```bash
make kaggle-load
python -m src.transform
make dw-load
```

## Star schema

```
dim_member   ──┐
dim_provider ──┼──► fact_claims
dim_date     ──┘

summary_kpis          (pre-aggregated KPIs)
summary_monthly       (monthly claim trends)
summary_loss_ratio    (loss ratios by region/network)
summary_network       (in/out-of-network utilisation)
```

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd

pd.set_option('display.float_format', '{:,.2f}'.format)

DW_PATH = '../outputs/insurance_dw.duckdb'
conn = duckdb.connect(DW_PATH, read_only=True)
print('Tables:', [r[0] for r in conn.execute('SHOW TABLES').fetchall()])

## 1. Total paid claims by age band and region

Classic star schema join across `fact_claims`, `dim_member`, and `dim_date`.

In [ ]:
q1 = conn.execute("""
    SELECT
        m.age_band,
        m.region,
        COUNT(f.claim_id)          AS claims,
        SUM(f.paid_amount)         AS total_paid,
        AVG(f.paid_amount)         AS avg_paid
    FROM fact_claims f
    JOIN dim_member  m ON f.member_id  = m.member_id
    GROUP BY m.age_band, m.region
    ORDER BY total_paid DESC
""").df()
q1

## 2. Monthly claim volume and spend trend

In [ ]:
q2 = conn.execute("""
    SELECT
        d.year,
        d.month_num,
        d.month_name,
        COUNT(f.claim_id)  AS claims,
        SUM(f.paid_amount) AS total_paid
    FROM fact_claims f
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY d.year, d.month_num, d.month_name
    ORDER BY d.year, d.month_num
""").df()
q2

## 3. Loss ratio by region

Loss ratio = paid / billed × 100. Pre-computed in `summary_loss_ratio`.

In [ ]:
q3 = conn.execute("""
    SELECT member_region, in_network, loss_ratio_pct, allowed_ratio_pct, claims
    FROM summary_loss_ratio
    ORDER BY loss_ratio_pct DESC
""").df()
q3

## 4. In-network vs out-of-network utilisation

In [ ]:
q4 = conn.execute("""
    SELECT
        p.in_network,
        COUNT(f.claim_id)          AS claims,
        SUM(f.paid_amount)         AS total_paid,
        ROUND(100.0 * COUNT(f.claim_id) / SUM(COUNT(f.claim_id)) OVER (), 2) AS pct_of_claims
    FROM fact_claims f
    JOIN dim_provider p ON f.provider_id = p.provider_id
    GROUP BY p.in_network
    ORDER BY p.in_network DESC
""").df()
q4

## 5. Top 10 diagnosis codes by total paid

In [ ]:
q5 = conn.execute("""
    SELECT
        diagnosis_code,
        COUNT(*) AS claims,
        SUM(paid_amount)  AS total_paid,
        AVG(paid_amount)  AS avg_paid
    FROM fact_claims
    GROUP BY diagnosis_code
    ORDER BY total_paid DESC
    LIMIT 10
""").df()
q5

## 6. Quarter-over-quarter spend comparison

In [ ]:
q6 = conn.execute("""
    SELECT
        d.year,
        d.quarter,
        COUNT(f.claim_id)          AS claims,
        SUM(f.paid_amount)         AS total_paid,
        SUM(f.billed_amount)       AS total_billed,
        ROUND(100.0 * SUM(f.paid_amount) / NULLIF(SUM(f.billed_amount), 0), 2) AS loss_ratio_pct
    FROM fact_claims f
    JOIN dim_date d ON f.date_key = d.date_key
    GROUP BY d.year, d.quarter
    ORDER BY d.year, d.quarter
""").df()
q6

## 7. DW Quality Report

Load the last-written QA report from `build/reports/dw_quality.json`.

In [ ]:
import json, pathlib
report_path = pathlib.Path('../build/reports/dw_quality.json')
if report_path.exists():
    report = json.loads(report_path.read_text())
    print('Generated at:', report['generated_at'])
    pd.DataFrame([report['checks']]).T.rename(columns={0: 'value'})
else:
    print('No QA report found. Run: make dw-load')

In [ ]:
conn.close()
print('Connection closed.')